In [ ]:
import tensorflow as tf

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
import math
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf, time
import keras

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, average_precision_score, fbeta_score)

In [ ]:
with tf.device("/GPU:0"):
    a = tf.random.normal((6000, 6000))
    t = time.time()
    for _ in range(20):
        b = tf.matmul(a, a)
    _ = b.numpy()
    print(f"GPU: {time.time() - t:.2f} s")

In [ ]:
DATA_DIR = Path("casting_512x512")
CLASS_TO_IDX = {"ok_front": 0, "def_front": 1}
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
SEED = 42
SPLIT_DIR = Path("splits")
REQUIRED_COLS = {"filepath", "label"}

In [ ]:
def collect_filepaths(data_dir: Path, class_to_idx: dict) -> pd.DataFrame:
    """Scan class subfolders and return a DataFrame with filepath, class name and label."""
    records = []
    for class_name, label in class_to_idx.items():
        class_dir = data_dir / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f"Class folder not found: {class_dir}")
        for fp in sorted(class_dir.iterdir()):  # sorted -> reproducible order
            if fp.suffix.lower() in IMG_EXTENSIONS:
                records.append({"filepath": str(fp), "class_name": class_name, "label": label})
    return pd.DataFrame(records)

In [ ]:
def show_class_distribution(df: pd.DataFrame, title: str = "Dataset") -> None:
    """Print absolute and relative class counts."""
    counts = df["class_name"].value_counts()
    ratios = df["class_name"].value_counts(normalize=True).round(3)
    print(f"--- {title} (n={len(df)}) ---")
    print(pd.DataFrame({"count": counts, "ratio": ratios}), "\n")

In [ ]:
df = collect_filepaths(DATA_DIR, CLASS_TO_IDX)
show_class_distribution(df, "Full dataset")

In [ ]:
def stratified_train_val_test_split(
    df: pd.DataFrame,
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    label_col: str = "label",
    seed: int = SEED,
):
    """Stratified split into train/val/test using two successive splits."""
    if abs(train_ratio + val_ratio + test_ratio - 1.0) > 1e-9:
        raise ValueError("Ratios must sum to 1.0")

    train_df, temp_df = train_test_split(
        df,
        test_size=(val_ratio + test_ratio),
        stratify=df[label_col],
        random_state=seed,
    )
    # share of the temp set that becomes the test set
    test_share = test_ratio / (val_ratio + test_ratio)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=test_share,
        stratify=temp_df[label_col],
        random_state=seed,
    )
    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )

In [ ]:
train_df, val_df, test_df = stratified_train_val_test_split(df)

for name, part in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    show_class_distribution(part, name)

In [ ]:
def verify_split(train_df, val_df, test_df, total_expected: int) -> None:
    """Sanity checks: no overlap between splits and no lost samples."""
    sets = {
        "train": set(train_df["filepath"]),
        "val": set(val_df["filepath"]),
        "test": set(test_df["filepath"]),
    }
    assert not (sets["train"] & sets["val"]), "Leakage: train/val overlap"
    assert not (sets["train"] & sets["test"]), "Leakage: train/test overlap"
    assert not (sets["val"] & sets["test"]), "Leakage: val/test overlap"
    assert sum(len(s) for s in sets.values()) == total_expected, "Samples lost or duplicated"
    print("Split verified: no overlap, no lost samples.")

In [ ]:
verify_split(train_df, val_df, test_df, total_expected=len(df))

In [ ]:
def save_split_manifests(train_df, val_df, test_df, out_dir: Path = SPLIT_DIR) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
        part.to_csv(out_dir / f"{name}.csv", index=False)
    print(f"Manifests saved to {out_dir.resolve()}")

In [ ]:
save_split_manifests(train_df, val_df, test_df)

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
def load_and_preprocess(filepath: tf.Tensor, label: tf.Tensor, img_size=IMG_SIZE):
    """Deterministic preprocessing: read -> grayscale -> resize -> scale to [0, 1]."""
    img = tf.io.read_file(filepath)
    img = tf.io.decode_image(img, channels=1, expand_animations=False)  # (H, W, 1)
    img = tf.image.resize(img, img_size)
    img = tf.cast(img, tf.float32) / 255.0
    img.set_shape((*img_size, 1))  # explicit shape helps downstream layers
    return img, tf.cast(label, tf.float32)

In [ ]:
def build_augmenter(
    rotation_factor: float = 0.05,    # fraction of 2*pi -> about +/-18 degrees
    brightness_factor: float = 0.1,   # +/-10% of the [0, 1] range
    contrast_factor: float = 0.1,     # +/-10%
    seed: int = 42,
) -> keras.Sequential:
    """Random geometric + photometric augmentations for [0, 1] grayscale images."""
    return keras.Sequential(
                            [
                                keras.layers.RandomFlip("horizontal_and_vertical", seed=seed),
                                keras.layers.RandomRotation(rotation_factor, fill_mode="reflect", seed=seed),
                                keras.layers.RandomBrightness(brightness_factor, value_range=(0.0, 1.0), seed=seed),
                                keras.layers.RandomContrast(contrast_factor, seed=seed),
                            ],
                            name="augmenter",
                           )

In [ ]:
def make_dataset(
    df: pd.DataFrame,
    training: bool,
    batch_size: int = BATCH_SIZE,
    img_size=IMG_SIZE,
    augmenter: keras.Sequential | None = None,
    seed: int = 42,
) -> tf.data.Dataset:
    """Build a tf.data pipeline. Shuffling and augmentation are applied only if training=True."""
    ds = tf.data.Dataset.from_tensor_slices((df["filepath"].values, df["label"].values))
    ds = ds.map(lambda p, y: load_and_preprocess(p, y, img_size), num_parallel_calls=AUTOTUNE)
    ds = ds.cache()

    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=seed, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size)

    if training and augmenter is not None:
        ds = ds.map(
            lambda x, y: (tf.clip_by_value(augmenter(x, training=True), 0.0, 1.0), y),
            num_parallel_calls=AUTOTUNE,
        )

    return ds.prefetch(AUTOTUNE)


def build_all_datasets(train_df, val_df, test_df, augmenter=None):
    train_ds = make_dataset(train_df, training=True, augmenter=augmenter)
    val_ds = make_dataset(val_df, training=False)    # no shuffle, no augmentation
    test_ds = make_dataset(test_df, training=False)  # no shuffle, no augmentation
    return train_ds, val_ds, test_ds

In [ ]:
augmenter = build_augmenter()
train_ds, val_ds, test_ds = build_all_datasets(train_df, val_df, test_df, augmenter)

In [ ]:
# def inspect_dataset(ds: tf.data.Dataset, name: str) -> None:
#     x, y = next(iter(ds))
#     print(f"{name}: x{tuple(x.shape)} {x.dtype.name} "
#           f"range=[{float(tf.reduce_min(x)):.3f}, {float(tf.reduce_max(x)):.3f}] | "
#           f"y{tuple(y.shape)} mean(defect share)={float(tf.reduce_mean(y)):.2f}")


# inspect_dataset(train_ds, "train")
# inspect_dataset(val_ds, "val")
# inspect_dataset(test_ds, "test")

In [ ]:
def plot_augmentations(df, augmenter, row_idx: int = None, n: int = 7, img_size=IMG_SIZE):
    """Show one original image next to n random augmentations of it."""
    row = df.iloc[row_idx]
    img, _ = load_and_preprocess(tf.constant(row["filepath"]), tf.constant(row["label"]), img_size)
    batch = tf.repeat(img[tf.newaxis], n, axis=0)   # n copies -> n different random draws
    aug = augmenter(batch, training=True)

    fig, axes = plt.subplots(1, n + 1, figsize=(2.6 * (n + 1), 3))
    axes[0].imshow(img[..., 0], cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("original")
    for i in range(n):
        axes[i + 1].imshow(aug[i, ..., 0], cmap="gray", vmin=0, vmax=1)
        axes[i + 1].set_title(f"aug {i + 1}")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"{row['class_name']} | {row['filepath'].split('/')[-1]}")
    plt.tight_layout()
    plt.show()

In [ ]:
# one healthy and one defective example from the train split
plot_augmentations(train_df, augmenter, row_idx=train_df.index[train_df["label"] == 0][0])
plot_augmentations(train_df, augmenter, row_idx=train_df.index[train_df["label"] == 1][0])

In [ ]:
def build_transfer_model(img_size=IMG_SIZE, dropout: float = 0.3):
    """MobileNetV2 (frozen) + small binary classification head. Input: (H, W, 1) in [0, 1]."""
    inputs = keras.Input(shape=(*img_size, 1), name="gray_input")

    x = keras.layers.Concatenate(name="gray_to_rgb")([inputs, inputs, inputs])    # 1 -> 3 channels
    x = keras.layers.Rescaling(scale=2.0, offset=-1.0, name="to_minus1_1")(x)     # [0,1] -> [-1,1]

    base = keras.applications.MobileNetV2(
        include_top=False, weights="imagenet", input_shape=(*img_size, 3)
    )
    base.trainable = False                  # phase 1: frozen
    x = base(x, training=False)             # keep BatchNorm layers in inference mode

    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(dropout)(x)
    outputs = keras.layers.Dense(1, activation="sigmoid", name="defect_prob")(x)

    return keras.Model(inputs, outputs, name="mobilenetv2_casting"), base


model, base_model = build_transfer_model()
model.summary()

In [ ]:
class WarmupCosine(keras.optimizers.schedules.LearningRateSchedule):
    """Linear warmup to peak_lr, then cosine decay to final_lr."""

    def __init__(self, peak_lr: float, total_steps: int, warmup_steps: int, final_lr: float = 1e-6):
        super().__init__()
        self.peak_lr = float(peak_lr)
        self.total_steps = int(total_steps)
        self.warmup_steps = int(warmup_steps)
        self.final_lr = float(final_lr)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup = tf.cast(self.warmup_steps, tf.float32)
        total = tf.cast(self.total_steps, tf.float32)

        warm_lr = self.peak_lr * (step + 1.0) / tf.maximum(1.0, warmup)
        progress = tf.clip_by_value((step - warmup) / tf.maximum(1.0, total - warmup), 0.0, 1.0)
        cos_lr = self.final_lr + 0.5 * (self.peak_lr - self.final_lr) * (1.0 + tf.cos(math.pi * progress))
        return tf.where(step < warmup, warm_lr, cos_lr)

    def get_config(self):
        return dict(peak_lr=self.peak_lr, total_steps=self.total_steps,
                    warmup_steps=self.warmup_steps, final_lr=self.final_lr)


def plot_schedule(schedule, total_steps: int) -> None:
    lrs = [float(schedule(s)) for s in range(total_steps)]
    plt.figure(figsize=(6, 3))
    plt.plot(lrs)
    plt.xlabel("step"); plt.ylabel("learning rate"); plt.title("Warmup + cosine decay")
    plt.tight_layout(); plt.show()

In [ ]:
EPOCHS_P1 = 15
steps_per_epoch = math.ceil(len(train_df) / BATCH_SIZE)

lr_schedule_p1 = WarmupCosine(
    peak_lr=1e-3,                              # fine for a fresh head on a frozen base
    total_steps=EPOCHS_P1 * steps_per_epoch,
    warmup_steps=2 * steps_per_epoch,          # 2 epochs of warmup
)
plot_schedule(lr_schedule_p1, EPOCHS_P1 * steps_per_epoch)


def compute_class_weights(df: pd.DataFrame, label_col: str = "label") -> dict:
    """Inverse-frequency class weights: total / (n_classes * count)."""
    counts = df[label_col].value_counts()
    return {int(c): len(df) / (len(counts) * n) for c, n in counts.items()}


def compile_model(model, lr) -> None:
    model.compile(
        jit_compile=False,
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="acc"),
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )

# def compare_schedules(schedules: dict, total_steps: int, log_y: bool = True) -> None:
#     plt.figure(figsize=(7, 3.5))
#     for name, sched in schedules.items():
#         plt.plot([float(sched(s)) for s in range(total_steps)], label=name)
#     if log_y:
#         plt.yscale("log")
#     plt.xlabel("step"); plt.ylabel("learning rate"); plt.legend(); plt.tight_layout(); plt.show()

def make_callbacks(ckpt_path: str, patience: int = 5):
    return [
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_auc", mode="max", save_best_only=True),
        keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=patience,
                                      restore_best_weights=True),
    ]

In [ ]:
class_weights = compute_class_weights(train_df)
print("class weights:", class_weights)

compile_model(model, lr_schedule_p1)
history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_P1,
    class_weight=class_weights,
    callbacks=make_callbacks("best_phase1.keras"),
)

In [ ]:
def plot_history(history, metrics=("loss", "auc", "recall", "precision")) -> None:
    """Plot train vs. validation curves for the given metrics."""
    fig, axes = plt.subplots(1, len(metrics), figsize=(4.5 * len(metrics), 3.2))
    for ax, m in zip(axes, metrics):
        ax.plot(history.history[m], label="train")
        ax.plot(history.history[f"val_{m}"], label="val")
        ax.set_title(m)
        ax.set_xlabel("epoch")
        ax.legend()
    plt.tight_layout()
    plt.show()


plot_history(history_p1)

In [ ]:
def set_finetune_layers(base_model, n_unfreeze: int = 30) -> None:
    """Unfreeze the top n layers of the base model, but keep all BatchNorm layers frozen."""
    base_model.trainable = True
    for layer in base_model.layers[:-n_unfreeze]:
        layer.trainable = False
    for layer in base_model.layers[-n_unfreeze:]:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False
    n_trainable = sum(l.trainable for l in base_model.layers)
    print(f"Trainable layers in base: {n_trainable} / {len(base_model.layers)}")


# model currently holds the best phase-1 weights (EarlyStopping restored them)
set_finetune_layers(base_model, n_unfreeze=30)

EPOCHS_P2 = 25
lr_schedule_p2 = WarmupCosine(
    peak_lr=5e-5,
    total_steps=EPOCHS_P2 * steps_per_epoch,
    warmup_steps=steps_per_epoch,       # 1 epoch warmup
    final_lr=1e-7,
)
plot_schedule(lr_schedule_p2, EPOCHS_P2 * steps_per_epoch)

compile_model(model, lr_schedule_p2)    # required after changing trainable flags
model.summary()                         # trainable params should now be well above 1,281

history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_P2,
    class_weight=class_weights,
    callbacks=make_callbacks("best_phase2.keras", patience=6),
)
plot_history(history_p2)

In [ ]:
CLASS_NAMES = ["ok_front", "def_front"]   # index = label


def predict_probs(model, ds) -> np.ndarray:
    """P(defect) per image, in dataset order (val/test are not shuffled)."""
    return model.predict(ds, verbose=0).ravel()


def choose_threshold(y_true, y_prob, beta: float = 2.0, grid=np.linspace(0.05, 0.95, 91)):
    """Threshold maximizing F-beta; on ties, return the middle of the best plateau."""
    scores = np.array([fbeta_score(y_true, y_prob >= t, beta=beta, zero_division=0) for t in grid])
    best_ts = grid[scores >= scores.max() - 1e-9]
    return float(np.median(best_ts)), float(scores.max()), (float(best_ts.min()), float(best_ts.max()))


def evaluate_split(y_true, y_prob, threshold: float, name: str) -> None:
    y_pred = (y_prob >= threshold).astype(int)
    print(f"===== {name} | threshold = {threshold:.3f} =====")
    print("Confusion matrix (rows = true, cols = pred; order: ok, defect)")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))
    print(f"ROC-AUC: {roc_auc_score(y_true, y_prob):.4f} | "
          f"PR-AUC: {average_precision_score(y_true, y_prob):.4f}\n")

In [ ]:
# 1) threshold from validation only
y_val = val_df["label"].values
p_val = predict_probs(model, val_ds)

evaluate_split(y_val, p_val, 0.5, "VAL (default 0.5)")
thr, best_f2, (t_lo, t_hi) = choose_threshold(y_val, p_val)
print(f"Chosen threshold {thr:.3f} (F2={best_f2:.3f}, plateau {t_lo:.2f} to {t_hi:.2f})\n")
evaluate_split(y_val, p_val, thr, "VAL (chosen threshold)")

In [ ]:
# 2) test set, evaluated once
y_test = test_df["label"].values
p_test = predict_probs(model, test_ds)
evaluate_split(y_test, p_test, thr, "TEST")

In [ ]:
def show_misclassified(df, y_prob, threshold: float, max_n: int = 8, img_size=IMG_SIZE) -> None:
    pred = (y_prob >= threshold).astype(int)
    wrong = df.assign(prob=y_prob, pred=pred)[pred != df["label"].values]
    print(f"{len(wrong)} misclassified of {len(df)}")
    if wrong.empty:
        return
    wrong = wrong.head(max_n)
    fig, axes = plt.subplots(1, len(wrong), figsize=(3.2 * len(wrong), 3.6), squeeze=False)
    for ax, (_, r) in zip(axes[0], wrong.iterrows()):
        img, _ = load_and_preprocess(tf.constant(r["filepath"]), tf.constant(r["label"]), img_size)
        ax.imshow(img[..., 0], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"true: {CLASS_NAMES[int(r['label'])]}\nP(defect)={r['prob']:.2f}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
show_misclassified(test_df, p_test, thr)

In [ ]:
def save_model_and_metadata(model, threshold: float, img_size=IMG_SIZE, out_dir: str = "best_models") -> None:
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    model.save(f"{out_dir}/casting_mobilenetv2.keras")
    meta = {
        "class_to_idx": CLASS_TO_IDX,
        "threshold": threshold,
        "input": {"size": list(img_size), "channels": 1, "range": "[0, 1] grayscale"},
        "in_model_preprocessing": "gray->RGB (3x concat), then Rescaling to [-1, 1]",
        "tensorflow": tf.__version__,
        "keras": keras.__version__,
    }
    with open(f"{out_dir}/model_metadata.json", "w") as f:
        json.dump(meta, f, indent=2)
    print(f"Saved to {out_dir}/")

In [ ]:
save_model_and_metadata(model, thr)